# ============================================
# MODULE 1: EXPLORATORY DATA ANALYSIS AND VISUALIZATION
# ============================================
#
# Learning Objectives:
# - Perform comprehensive exploratory data analysis on power systems data
# - Create effective visualizations to understand electrical measurements
# - Identify patterns, trends, and anomalies in energy data
# - Understand relationships between electrical parameters
# - Communicate insights through professional visualizations
#
# Real-World Application:
# In power systems, understanding data patterns is crucial for:
# - Identifying peak demand periods for generation planning
# - Detecting unusual patterns that indicate equipment issues
# - Understanding correlations between voltage, current, and power
# - Communicating technical findings to stakeholders
# - Making data-driven decisions for grid operations
#
# Estimated Time: 3-4 hours
# ============================================

## Section 1: Import Libraries and Load Data

In [ ]:
# Import pandas for data manipulation and analysis
import pandas as pd

# Import numpy for numerical operations
import numpy as np

# Import matplotlib for creating visualizations
import matplotlib.pyplot as plt

# Import seaborn for enhanced statistical visualizations
import seaborn as sns

# Import plotly for interactive visualizations
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Import datetime for time-based operations
from datetime import datetime, timedelta

# Import statistics module for additional statistical functions
import statistics

# Import scipy for statistical tests
from scipy import stats

# Import warnings to manage warning messages
import warnings
warnings.filterwarnings('ignore')

# Set pandas display options for better output
pd.set_option('display.max_columns', None)
pd.set_option('display.precision', 3)

# Set seaborn style for professional-looking plots
sns.set_style('whitegrid')
sns.set_palette('husl')

# Set default matplotlib figure size
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['font.size'] = 10

print("Libraries imported successfully!")

## Section 2: Generate Comprehensive Power System Dataset

We'll create a more comprehensive dataset with additional features for EDA practice.

In [ ]:
# Set random seed for reproducibility
np.random.seed(42)

# Generate 2 months of hourly data (more data for better pattern analysis)
n_records = 1440  # 60 days * 24 hours

# Create timestamp range starting from January 1, 2023
date_range = pd.date_range(start='2023-01-01', periods=n_records, freq='H')

# Create base load with realistic daily and weekly patterns
# Extract hour of day (0-23) for daily patterns
hours = date_range.hour

# Extract day of week (0=Monday, 6=Sunday) for weekly patterns
day_of_week = date_range.dayofweek

# Create daily load pattern (higher during day, lower at night)
# Using sine wave to create smooth daily cycle
# Peak occurs around 2 PM (hour 14), minimum around 2 AM (hour 2)
daily_pattern = 100 + 50 * np.sin((hours - 6) * np.pi / 12)

# Create weekly pattern (lower on weekends)
# Weekdays (0-4) have higher load than weekends (5-6)
weekly_pattern = np.where(day_of_week < 5, 1.0, 0.85)

# Combine patterns to create base load in MW
base_load = daily_pattern * weekly_pattern

# Add random variations to make it realistic
# Real power systems have stochastic load variations
load_mw = base_load + np.random.normal(0, 10, n_records)

# Generate voltage based on load (voltage drops slightly with higher load)
# Nominal voltage is 230 kV
# Voltage regulation: higher load causes slight voltage drop
voltage_kv = 230 - (load_mw - load_mw.mean()) * 0.02 + np.random.normal(0, 2, n_records)

# Generate current based on load and voltage
# Using P = √3 × V × I relationship, solve for I
# I = P / (√3 × V)
current_a = (load_mw * 1000) / (np.sqrt(3) * voltage_kv) + np.random.normal(0, 20, n_records)

# Generate frequency with small variations
# Grid frequency should be very stable around 60 Hz
# Small deviations indicate generation-load imbalance
frequency_hz = 60.0 + np.random.normal(0, 0.02, n_records)

# Generate power factor (typical range: 0.85 to 0.95)
# Lower power factor indicates more reactive power
# Industrial loads typically have lower power factor
power_factor = np.random.uniform(0.85, 0.95, n_records)

# Calculate reactive power based on real power and power factor
# Q = P × tan(arccos(PF))
reactive_power_mvar = load_mw * np.tan(np.arccos(power_factor))

# Calculate apparent power
# S = P / PF
apparent_power_mva = load_mw / power_factor

# Generate weather data (affects load patterns)
# Temperature in Celsius with seasonal variation
# January-February: colder temperatures
day_of_year = date_range.dayofyear
temperature_c = 15 + 10 * np.sin((day_of_year - 15) * np.pi / 182.5) + np.random.normal(0, 3, n_records)

# Humidity percentage (affects electrical insulation)
humidity_percent = np.random.uniform(40, 80, n_records)

# Create the comprehensive DataFrame
df = pd.DataFrame({
    'timestamp': date_range,
    'load_mw': load_mw,
    'voltage_kv': voltage_kv,
    'current_a': current_a,
    'frequency_hz': frequency_hz,
    'power_factor': power_factor,
    'reactive_power_mvar': reactive_power_mvar,
    'apparent_power_mva': apparent_power_mva,
    'temperature_c': temperature_c,
    'humidity_percent': humidity_percent
})

# Add temporal features for analysis
# Hour of day (0-23)
df['hour'] = df['timestamp'].dt.hour

# Day of week (0=Monday, 6=Sunday)
df['day_of_week'] = df['timestamp'].dt.dayofweek

# Day name for better readability
df['day_name'] = df['timestamp'].dt.day_name()

# Is weekend flag (useful for pattern analysis)
df['is_weekend'] = (df['day_of_week'] >= 5).astype(int)

# Month
df['month'] = df['timestamp'].dt.month

# Month name
df['month_name'] = df['timestamp'].dt.month_name()

print(f"Generated {len(df)} power system records")
print(f"Date range: {df['timestamp'].min()} to {df['timestamp'].max()}")
print(f"Total columns: {len(df.columns)}")

## Section 3: Initial Data Overview

In [ ]:
# Display first few rows to understand data structure
print("First 10 rows of the dataset:")
print(df.head(10))

In [ ]:
# Display last few rows to check data completeness
print("Last 10 rows of the dataset:")
print(df.tail(10))

In [ ]:
# Get detailed information about the dataset
print("Dataset Information:")
print(df.info())

# Display dataset shape
print(f"\nDataset shape: {df.shape[0]} rows × {df.shape[1]} columns")

In [ ]:
# Get column names and data types
print("Column names and data types:")
print(df.dtypes)

## Section 4: Statistical Summary Analysis

In [ ]:
# Get comprehensive statistical summary of numerical columns
# This includes count, mean, std, min, quartiles, and max
print("Statistical Summary of Power System Parameters:")
print("=" * 80)

# Select only numerical columns for statistical analysis
numerical_cols = df.select_dtypes(include=[np.number]).columns
summary = df[numerical_cols].describe()

# Display with transposed view for better readability
print(summary.T)

In [ ]:
# Calculate additional statistical measures
print("\nAdditional Statistical Measures:")
print("=" * 80)

# Focus on key electrical parameters
key_params = ['load_mw', 'voltage_kv', 'current_a', 'frequency_hz', 'power_factor']

for param in key_params:
    print(f"\n{param.upper()}:")
    
    # Calculate median (50th percentile)
    # Median is less sensitive to outliers than mean
    median = df[param].median()
    
    # Calculate mode (most frequent value)
    # For continuous data, we use the value that appears most in bins
    mode = df[param].mode()[0] if len(df[param].mode()) > 0 else 'N/A'
    
    # Calculate variance (spread of data)
    variance = df[param].var()
    
    # Calculate coefficient of variation (relative variability)
    # CV = (std / mean) × 100%
    # Lower CV indicates more stable measurements
    cv = (df[param].std() / df[param].mean()) * 100
    
    # Calculate skewness (asymmetry of distribution)
    # Skewness = 0: symmetric distribution
    # Skewness > 0: right-tailed distribution
    # Skewness < 0: left-tailed distribution
    skewness = df[param].skew()
    
    # Calculate kurtosis (tailedness of distribution)
    # Kurtosis = 3: normal distribution
    # Kurtosis > 3: heavy tails (more outliers)
    # Kurtosis < 3: light tails (fewer outliers)
    kurtosis = df[param].kurtosis()
    
    print(f"  Median: {median:.3f}")
    print(f"  Mode: {mode if isinstance(mode, str) else f'{mode:.3f}'}")
    print(f"  Variance: {variance:.3f}")
    print(f"  Coefficient of Variation: {cv:.2f}%")
    print(f"  Skewness: {skewness:.3f}")
    print(f"  Kurtosis: {kurtosis:.3f}")

## Section 5: Distribution Analysis with Visualizations

In [ ]:
# Create histograms to visualize distribution of key parameters
# Histograms show the frequency of values in different ranges

fig, axes = plt.subplots(3, 2, figsize=(16, 14))
fig.suptitle('Distribution of Power System Parameters', fontsize=16, fontweight='bold', y=0.995)

# Define parameters to plot
params_to_plot = [
    ('load_mw', 'Load (MW)', 'blue'),
    ('voltage_kv', 'Voltage (kV)', 'green'),
    ('current_a', 'Current (A)', 'orange'),
    ('frequency_hz', 'Frequency (Hz)', 'red'),
    ('power_factor', 'Power Factor', 'purple'),
    ('temperature_c', 'Temperature (°C)', 'brown')
]

# Create histogram for each parameter
for idx, (param, label, color) in enumerate(params_to_plot):
    # Calculate subplot position
    row = idx // 2
    col = idx % 2
    
    # Create histogram with 50 bins
    # bins=50 provides good granularity for continuous data
    axes[row, col].hist(df[param], bins=50, color=color, alpha=0.7, edgecolor='black', linewidth=0.5)
    
    # Add vertical line for mean
    mean_val = df[param].mean()
    axes[row, col].axvline(mean_val, color='red', linestyle='--', linewidth=2, label=f'Mean: {mean_val:.2f}')
    
    # Add vertical line for median
    median_val = df[param].median()
    axes[row, col].axvline(median_val, color='green', linestyle='--', linewidth=2, label=f'Median: {median_val:.2f}')
    
    # Set labels and title
    axes[row, col].set_xlabel(label, fontsize=11, fontweight='bold')
    axes[row, col].set_ylabel('Frequency', fontsize=11, fontweight='bold')
    axes[row, col].set_title(f'{label} Distribution', fontsize=12, fontweight='bold')
    
    # Add grid for better readability
    axes[row, col].grid(True, alpha=0.3)
    
    # Add legend
    axes[row, col].legend()

plt.tight_layout()
plt.show()

print("Distribution analysis complete")

In [ ]:
# Create box plots to identify outliers and quartiles
# Box plots show: median, quartiles (Q1, Q3), and outliers

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
fig.suptitle('Box Plots: Outlier Detection in Power System Parameters', fontsize=16, fontweight='bold')

params_to_plot = [
    ('load_mw', 'Load (MW)'),
    ('voltage_kv', 'Voltage (kV)'),
    ('current_a', 'Current (A)'),
    ('frequency_hz', 'Frequency (Hz)'),
    ('power_factor', 'Power Factor'),
    ('reactive_power_mvar', 'Reactive Power (MVAr)')
]

for idx, (param, label) in enumerate(params_to_plot):
    row = idx // 3
    col = idx % 3
    
    # Create box plot
    # showmeans=True displays the mean as a separate marker
    bp = axes[row, col].boxplot(df[param], vert=True, patch_artist=True, showmeans=True,
                                  meanprops=dict(marker='D', markerfacecolor='red', markersize=8))
    
    # Color the box
    bp['boxes'][0].set_facecolor('lightblue')
    bp['boxes'][0].set_alpha(0.7)
    
    # Set labels
    axes[row, col].set_ylabel(label, fontsize=11, fontweight='bold')
    axes[row, col].set_title(f'{label} Box Plot', fontsize=12, fontweight='bold')
    axes[row, col].grid(True, alpha=0.3, axis='y')
    
    # Calculate and display statistics on the plot
    q1 = df[param].quantile(0.25)
    q3 = df[param].quantile(0.75)
    iqr = q3 - q1
    
    # Add text box with statistics
    stats_text = f'Q1: {q1:.2f}\nMedian: {df[param].median():.2f}\nQ3: {q3:.2f}\nIQR: {iqr:.2f}'
    axes[row, col].text(1.15, df[param].median(), stats_text, fontsize=9,
                        bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

plt.tight_layout()
plt.show()

print("Box plot analysis complete")
print("\nNote: Points outside whiskers are potential outliers")

## Section 6: Correlation Analysis

Understanding relationships between electrical parameters is crucial for power system analysis.

In [ ]:
# Calculate correlation matrix for all numerical variables
# Correlation measures linear relationship between variables
# Correlation = 1: perfect positive correlation
# Correlation = -1: perfect negative correlation
# Correlation = 0: no linear correlation

# Select key electrical parameters for correlation analysis
electrical_params = ['load_mw', 'voltage_kv', 'current_a', 'frequency_hz', 
                     'power_factor', 'reactive_power_mvar', 'apparent_power_mva', 
                     'temperature_c', 'humidity_percent']

# Calculate Pearson correlation coefficient
# This measures linear relationships between variables
correlation_matrix = df[electrical_params].corr()

print("Correlation Matrix:")
print(correlation_matrix.round(3))

In [ ]:
# Visualize correlation matrix with a heatmap
# Heatmap uses color intensity to show correlation strength

plt.figure(figsize=(14, 10))

# Create heatmap with annotations
# annot=True displays correlation values in each cell
# fmt='.2f' formats numbers to 2 decimal places
# cmap='coolwarm' uses blue for negative, red for positive correlations
# center=0 ensures white color represents zero correlation
sns.heatmap(correlation_matrix, annot=True, fmt='.2f', cmap='coolwarm', 
            center=0, square=True, linewidths=1, cbar_kws={'label': 'Correlation Coefficient'},
            vmin=-1, vmax=1)

plt.title('Correlation Heatmap: Power System Parameters', fontsize=16, fontweight='bold', pad=20)
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

print("\nKey Correlations to Note:")
print("- Load and Current should be highly correlated (both measure power consumption)")
print("- Load and Voltage may have slight negative correlation (voltage drop under load)")
print("- Reactive and Apparent Power should correlate with Real Power")

In [ ]:
# Find and display strongest correlations
# This helps identify most important relationships

# Extract upper triangle of correlation matrix (avoid duplicates)
# np.triu creates upper triangular matrix
# k=1 excludes diagonal (self-correlations)
upper_triangle = correlation_matrix.where(np.triu(np.ones(correlation_matrix.shape), k=1).astype(bool))

# Find pairs with highest absolute correlation
# Stack converts matrix to series, dropna removes NaN values
correlations = upper_triangle.stack().reset_index()
correlations.columns = ['Variable 1', 'Variable 2', 'Correlation']

# Sort by absolute correlation value
correlations['Abs_Correlation'] = correlations['Correlation'].abs()
correlations = correlations.sort_values('Abs_Correlation', ascending=False)

print("\nTop 10 Strongest Correlations:")
print("=" * 80)
print(correlations.head(10)[['Variable 1', 'Variable 2', 'Correlation']].to_string(index=False))

## Section 7: Time Series Analysis and Patterns

In [ ]:
# Plot load over time to identify temporal patterns
# Time series plots reveal trends, cycles, and anomalies

fig, axes = plt.subplots(3, 1, figsize=(16, 12))
fig.suptitle('Time Series Analysis: Load Patterns', fontsize=16, fontweight='bold')

# Plot 1: Complete time series
axes[0].plot(df['timestamp'], df['load_mw'], linewidth=0.8, color='blue', alpha=0.7)
axes[0].set_title('Complete Load Time Series (2 Months)', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Load (MW)', fontsize=11, fontweight='bold')
axes[0].grid(True, alpha=0.3)

# Add horizontal line for mean load
axes[0].axhline(df['load_mw'].mean(), color='red', linestyle='--', 
                label=f"Mean: {df['load_mw'].mean():.2f} MW", linewidth=2)
axes[0].legend()

# Plot 2: One week detail
# Zoom into one week to see daily patterns more clearly
one_week = df[df['timestamp'] < df['timestamp'].min() + pd.Timedelta(days=7)]
axes[1].plot(one_week['timestamp'], one_week['load_mw'], linewidth=1.5, 
             color='green', marker='o', markersize=2, alpha=0.7)
axes[1].set_title('One Week Detail (Daily Patterns Visible)', fontsize=13, fontweight='bold')
axes[1].set_ylabel('Load (MW)', fontsize=11, fontweight='bold')
axes[1].grid(True, alpha=0.3)

# Highlight weekends with shaded regions
for idx, row in one_week.iterrows():
    if row['is_weekend'] == 1:
        axes[1].axvspan(row['timestamp'], row['timestamp'] + pd.Timedelta(hours=1), 
                        alpha=0.2, color='gray')

# Plot 3: One day detail
# Zoom into one day to see hourly variations
one_day = df[df['timestamp'].dt.date == df['timestamp'].dt.date.min()]
axes[2].plot(one_day['timestamp'], one_day['load_mw'], linewidth=2, 
             color='purple', marker='o', markersize=6, alpha=0.7)
axes[2].set_title('One Day Detail (Hourly Variations)', fontsize=13, fontweight='bold')
axes[2].set_xlabel('Time', fontsize=11, fontweight='bold')
axes[2].set_ylabel('Load (MW)', fontsize=11, fontweight='bold')
axes[2].grid(True, alpha=0.3)

# Mark peak and minimum load hours
peak_hour = one_day.loc[one_day['load_mw'].idxmax()]
min_hour = one_day.loc[one_day['load_mw'].idxmin()]
axes[2].scatter(peak_hour['timestamp'], peak_hour['load_mw'], 
                color='red', s=200, zorder=5, label='Peak Load')
axes[2].scatter(min_hour['timestamp'], min_hour['load_mw'], 
                color='blue', s=200, zorder=5, label='Minimum Load')
axes[2].legend()

plt.tight_layout()
plt.show()

print("Time series patterns identified")

In [ ]:
# Analyze hourly patterns (average load by hour of day)
# This reveals typical daily load curve

# Group by hour and calculate statistics
hourly_stats = df.groupby('hour')['load_mw'].agg([
    ('mean', 'mean'),
    ('std', 'std'),
    ('min', 'min'),
    ('max', 'max')
]).reset_index()

# Create visualization
fig, ax = plt.subplots(figsize=(16, 8))

# Plot mean load by hour with shaded standard deviation band
ax.plot(hourly_stats['hour'], hourly_stats['mean'], 
        linewidth=3, color='blue', marker='o', markersize=8, label='Mean Load')

# Add shaded region for ±1 standard deviation
# This shows typical variation around the mean
ax.fill_between(hourly_stats['hour'], 
                hourly_stats['mean'] - hourly_stats['std'],
                hourly_stats['mean'] + hourly_stats['std'],
                alpha=0.3, color='blue', label='±1 Std Dev')

# Plot min and max ranges
ax.plot(hourly_stats['hour'], hourly_stats['min'], 
        linewidth=1.5, color='green', linestyle='--', alpha=0.7, label='Minimum')
ax.plot(hourly_stats['hour'], hourly_stats['max'], 
        linewidth=1.5, color='red', linestyle='--', alpha=0.7, label='Maximum')

# Formatting
ax.set_xlabel('Hour of Day', fontsize=13, fontweight='bold')
ax.set_ylabel('Load (MW)', fontsize=13, fontweight='bold')
ax.set_title('Average Load Profile by Hour of Day', fontsize=15, fontweight='bold', pad=20)
ax.set_xticks(range(0, 24))
ax.grid(True, alpha=0.3)
ax.legend(fontsize=11)

# Add annotations for key hours
peak_hour_idx = hourly_stats['mean'].idxmax()
min_hour_idx = hourly_stats['mean'].idxmin()

ax.annotate(f'Peak Load Hour\n{hourly_stats.loc[peak_hour_idx, "mean"]:.2f} MW',
            xy=(hourly_stats.loc[peak_hour_idx, 'hour'], hourly_stats.loc[peak_hour_idx, 'mean']),
            xytext=(hourly_stats.loc[peak_hour_idx, 'hour'] + 2, hourly_stats.loc[peak_hour_idx, 'mean'] + 10),
            arrowprops=dict(arrowstyle='->', color='red', lw=2),
            fontsize=11, fontweight='bold', color='red',
            bbox=dict(boxstyle='round', facecolor='white', edgecolor='red', alpha=0.8))

ax.annotate(f'Minimum Load Hour\n{hourly_stats.loc[min_hour_idx, "mean"]:.2f} MW',
            xy=(hourly_stats.loc[min_hour_idx, 'hour'], hourly_stats.loc[min_hour_idx, 'mean']),
            xytext=(hourly_stats.loc[min_hour_idx, 'hour'] + 2, hourly_stats.loc[min_hour_idx, 'mean'] - 15),
            arrowprops=dict(arrowstyle='->', color='blue', lw=2),
            fontsize=11, fontweight='bold', color='blue',
            bbox=dict(boxstyle='round', facecolor='white', edgecolor='blue', alpha=0.8))

plt.tight_layout()
plt.show()

print("\nHourly Load Statistics:")
print(hourly_stats.round(2))

In [ ]:
# Analyze weekly patterns (weekday vs weekend)
# Compare load profiles between weekdays and weekends

# Separate weekday and weekend data
weekday_data = df[df['is_weekend'] == 0]
weekend_data = df[df['is_weekend'] == 1]

# Calculate hourly averages for each group
weekday_hourly = weekday_data.groupby('hour')['load_mw'].mean()
weekend_hourly = weekend_data.groupby('hour')['load_mw'].mean()

# Create comparison plot
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('Weekday vs Weekend Load Patterns', fontsize=16, fontweight='bold')

# Plot 1: Line comparison
axes[0].plot(weekday_hourly.index, weekday_hourly.values, 
             linewidth=3, color='blue', marker='o', markersize=7, label='Weekday')
axes[0].plot(weekend_hourly.index, weekend_hourly.values, 
             linewidth=3, color='orange', marker='s', markersize=7, label='Weekend')
axes[0].set_xlabel('Hour of Day', fontsize=12, fontweight='bold')
axes[0].set_ylabel('Average Load (MW)', fontsize=12, fontweight='bold')
axes[0].set_title('Load Profile Comparison', fontsize=13, fontweight='bold')
axes[0].set_xticks(range(0, 24))
axes[0].grid(True, alpha=0.3)
axes[0].legend(fontsize=12)

# Plot 2: Box plot comparison
# Create data structure for box plot
weekday_weekend_data = pd.DataFrame({
    'Weekday': weekday_data['load_mw'],
    'Weekend': weekend_data['load_mw']
})

bp = axes[1].boxplot([weekday_data['load_mw'], weekend_data['load_mw']], 
                      labels=['Weekday', 'Weekend'],
                      patch_artist=True, showmeans=True,
                      meanprops=dict(marker='D', markerfacecolor='red', markersize=10))

# Color the boxes
bp['boxes'][0].set_facecolor('lightblue')
bp['boxes'][1].set_facecolor('lightyellow')

axes[1].set_ylabel('Load (MW)', fontsize=12, fontweight='bold')
axes[1].set_title('Load Distribution Comparison', fontsize=13, fontweight='bold')
axes[1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

print("\nWeekday vs Weekend Load Statistics:")
print(f"Weekday Average: {weekday_data['load_mw'].mean():.2f} MW")
print(f"Weekend Average: {weekend_data['load_mw'].mean():.2f} MW")
print(f"Difference: {weekday_data['load_mw'].mean() - weekend_data['load_mw'].mean():.2f} MW")
print(f"Percentage Difference: {((weekday_data['load_mw'].mean() - weekend_data['load_mw'].mean()) / weekday_data['load_mw'].mean() * 100):.2f}%")

## Section 8: Scatter Plots and Relationship Analysis

In [ ]:
# Create scatter plots to visualize relationships between variables
# Scatter plots reveal linear and non-linear relationships

fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle('Scatter Plots: Relationships Between Power System Parameters', 
             fontsize=16, fontweight='bold')

# Plot 1: Load vs Current (should be strongly correlated)
axes[0, 0].scatter(df['load_mw'], df['current_a'], alpha=0.5, s=10, color='blue')
axes[0, 0].set_xlabel('Load (MW)', fontsize=11, fontweight='bold')
axes[0, 0].set_ylabel('Current (A)', fontsize=11, fontweight='bold')
axes[0, 0].set_title('Load vs Current', fontsize=12, fontweight='bold')
axes[0, 0].grid(True, alpha=0.3)

# Add trend line using numpy polyfit
# polyfit calculates best-fit line coefficients
z = np.polyfit(df['load_mw'], df['current_a'], 1)
p = np.poly1d(z)
axes[0, 0].plot(df['load_mw'], p(df['load_mw']), "r--", linewidth=2, label='Trend Line')
axes[0, 0].legend()

# Plot 2: Load vs Voltage (slight negative correlation expected)
axes[0, 1].scatter(df['load_mw'], df['voltage_kv'], alpha=0.5, s=10, color='green')
axes[0, 1].set_xlabel('Load (MW)', fontsize=11, fontweight='bold')
axes[0, 1].set_ylabel('Voltage (kV)', fontsize=11, fontweight='bold')
axes[0, 1].set_title('Load vs Voltage (Voltage Regulation)', fontsize=12, fontweight='bold')
axes[0, 1].grid(True, alpha=0.3)

z = np.polyfit(df['load_mw'], df['voltage_kv'], 1)
p = np.poly1d(z)
axes[0, 1].plot(df['load_mw'], p(df['load_mw']), "r--", linewidth=2, label='Trend Line')
axes[0, 1].legend()

# Plot 3: Temperature vs Load (correlation expected)
# Color points by time of day to show additional dimension
scatter = axes[1, 0].scatter(df['temperature_c'], df['load_mw'], 
                             c=df['hour'], cmap='viridis', alpha=0.6, s=15)
axes[1, 0].set_xlabel('Temperature (°C)', fontsize=11, fontweight='bold')
axes[1, 0].set_ylabel('Load (MW)', fontsize=11, fontweight='bold')
axes[1, 0].set_title('Temperature vs Load (colored by Hour)', fontsize=12, fontweight='bold')
axes[1, 0].grid(True, alpha=0.3)

# Add colorbar to show hour mapping
cbar = plt.colorbar(scatter, ax=axes[1, 0])
cbar.set_label('Hour of Day', fontsize=10)

# Plot 4: Power Factor vs Reactive Power
axes[1, 1].scatter(df['power_factor'], df['reactive_power_mvar'], 
                   alpha=0.5, s=10, color='purple')
axes[1, 1].set_xlabel('Power Factor', fontsize=11, fontweight='bold')
axes[1, 1].set_ylabel('Reactive Power (MVAr)', fontsize=11, fontweight='bold')
axes[1, 1].set_title('Power Factor vs Reactive Power', fontsize=12, fontweight='bold')
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("Scatter plot analysis complete")

## Section 9: Advanced Visualizations with Plotly (Interactive)

Interactive plots allow zooming, panning, and hovering for detailed exploration.

In [ ]:
# Create interactive time series plot with Plotly
# Interactive plots allow users to zoom, pan, and explore data dynamically

# Create figure with secondary y-axis
# This allows plotting two different scales on one chart
fig = make_subplots(
    rows=2, cols=1,
    subplot_titles=('Load and Voltage Over Time', 'Frequency Over Time'),
    vertical_spacing=0.12,
    specs=[[{"secondary_y": True}], [{"secondary_y": False}]]
)

# Add load trace on primary y-axis
fig.add_trace(
    go.Scatter(x=df['timestamp'], y=df['load_mw'], 
               name='Load (MW)', line=dict(color='blue', width=1)),
    row=1, col=1, secondary_y=False
)

# Add voltage trace on secondary y-axis
fig.add_trace(
    go.Scatter(x=df['timestamp'], y=df['voltage_kv'], 
               name='Voltage (kV)', line=dict(color='red', width=1)),
    row=1, col=1, secondary_y=True
)

# Add frequency trace
fig.add_trace(
    go.Scatter(x=df['timestamp'], y=df['frequency_hz'], 
               name='Frequency (Hz)', line=dict(color='green', width=1)),
    row=2, col=1
)

# Update axes labels
fig.update_xaxes(title_text="Time", row=2, col=1)
fig.update_yaxes(title_text="Load (MW)", row=1, col=1, secondary_y=False)
fig.update_yaxes(title_text="Voltage (kV)", row=1, col=1, secondary_y=True)
fig.update_yaxes(title_text="Frequency (Hz)", row=2, col=1)

# Update layout
fig.update_layout(
    height=700,
    title_text="Interactive Power System Time Series",
    hovermode='x unified',
    showlegend=True
)

fig.show()

print("Interactive plot created - hover over points to see values")

In [ ]:
# Create interactive 3D scatter plot
# 3D visualization reveals relationships between three variables

fig = px.scatter_3d(df, 
                    x='load_mw', 
                    y='voltage_kv', 
                    z='current_a',
                    color='temperature_c',
                    color_continuous_scale='Viridis',
                    labels={
                        'load_mw': 'Load (MW)',
                        'voltage_kv': 'Voltage (kV)',
                        'current_a': 'Current (A)',
                        'temperature_c': 'Temperature (°C)'
                    },
                    title='3D Scatter Plot: Load, Voltage, Current (colored by Temperature)',
                    opacity=0.7)

# Update layout for better view
fig.update_layout(
    scene=dict(
        xaxis_title='Load (MW)',
        yaxis_title='Voltage (kV)',
        zaxis_title='Current (A)'
    ),
    height=700
)

fig.show()

print("3D scatter plot created - rotate and zoom to explore")

## Section 10: Statistical Hypothesis Testing

Test if weekday and weekend loads are statistically different.

In [ ]:
# Perform t-test to determine if weekday and weekend loads are significantly different
# T-test compares means of two groups
# Null hypothesis: weekday and weekend loads have the same mean
# Alternative hypothesis: weekday and weekend loads have different means

from scipy.stats import ttest_ind

# Extract weekday and weekend load data
weekday_loads = df[df['is_weekend'] == 0]['load_mw']
weekend_loads = df[df['is_weekend'] == 1]['load_mw']

# Perform independent samples t-test
# equal_var=False uses Welch's t-test (doesn't assume equal variances)
t_statistic, p_value = ttest_ind(weekday_loads, weekend_loads, equal_var=False)

print("T-Test: Weekday vs Weekend Load")
print("=" * 60)
print(f"Weekday Load - Mean: {weekday_loads.mean():.2f} MW, Std: {weekday_loads.std():.2f} MW")
print(f"Weekend Load - Mean: {weekend_loads.mean():.2f} MW, Std: {weekend_loads.std():.2f} MW")
print(f"\nT-statistic: {t_statistic:.4f}")
print(f"P-value: {p_value:.6f}")

# Interpret results
# Typically, p < 0.05 indicates statistical significance
alpha = 0.05
if p_value < alpha:
    print(f"\nResult: REJECT null hypothesis (p < {alpha})")
    print("Conclusion: Weekday and weekend loads are SIGNIFICANTLY DIFFERENT")
else:
    print(f"\nResult: FAIL TO REJECT null hypothesis (p >= {alpha})")
    print("Conclusion: No significant difference between weekday and weekend loads")

# Calculate effect size (Cohen's d)
# Effect size measures practical significance (not just statistical)
# Cohen's d = (mean1 - mean2) / pooled_std
pooled_std = np.sqrt((weekday_loads.std()**2 + weekend_loads.std()**2) / 2)
cohens_d = (weekday_loads.mean() - weekend_loads.mean()) / pooled_std

print(f"\nEffect Size (Cohen's d): {cohens_d:.4f}")
if abs(cohens_d) < 0.2:
    print("Effect size interpretation: SMALL")
elif abs(cohens_d) < 0.5:
    print("Effect size interpretation: MEDIUM")
else:
    print("Effect size interpretation: LARGE")

## Section 11: Summary Report and Key Insights

In [ ]:
# Generate comprehensive EDA summary report
print("=" * 80)
print("EXPLORATORY DATA ANALYSIS - SUMMARY REPORT")
print("=" * 80)

print("\n1. DATASET OVERVIEW")
print("-" * 80)
print(f"Total Records: {len(df):,}")
print(f"Date Range: {df['timestamp'].min()} to {df['timestamp'].max()}")
print(f"Duration: {(df['timestamp'].max() - df['timestamp'].min()).days} days")
print(f"Number of Features: {len(df.columns)}")

print("\n2. LOAD STATISTICS")
print("-" * 80)
print(f"Average Load: {df['load_mw'].mean():.2f} MW")
print(f"Peak Load: {df['load_mw'].max():.2f} MW at {df.loc[df['load_mw'].idxmax(), 'timestamp']}")
print(f"Minimum Load: {df['load_mw'].min():.2f} MW at {df.loc[df['load_mw'].idxmin(), 'timestamp']}")
print(f"Load Range: {df['load_mw'].max() - df['load_mw'].min():.2f} MW")
print(f"Load Variability (CV): {(df['load_mw'].std() / df['load_mw'].mean() * 100):.2f}%")

print("\n3. TEMPORAL PATTERNS")
print("-" * 80)
peak_hour = df.groupby('hour')['load_mw'].mean().idxmax()
min_hour = df.groupby('hour')['load_mw'].mean().idxmin()
print(f"Peak Load Hour: {peak_hour}:00 ({df.groupby('hour')['load_mw'].mean().max():.2f} MW avg)")
print(f"Minimum Load Hour: {min_hour}:00 ({df.groupby('hour')['load_mw'].mean().min():.2f} MW avg)")
print(f"Weekday Average Load: {weekday_loads.mean():.2f} MW")
print(f"Weekend Average Load: {weekend_loads.mean():.2f} MW")
print(f"Weekday/Weekend Difference: {weekday_loads.mean() - weekend_loads.mean():.2f} MW ({((weekday_loads.mean() - weekend_loads.mean()) / weekday_loads.mean() * 100):.1f}%)")

print("\n4. POWER QUALITY METRICS")
print("-" * 80)
print(f"Average Voltage: {df['voltage_kv'].mean():.2f} kV (Nominal: 230 kV)")
print(f"Voltage Variation: {df['voltage_kv'].std():.2f} kV ({(df['voltage_kv'].std() / df['voltage_kv'].mean() * 100):.3f}%)")
print(f"Average Frequency: {df['frequency_hz'].mean():.4f} Hz (Nominal: 60.00 Hz)")
print(f"Frequency Variation: {df['frequency_hz'].std():.4f} Hz ({(df['frequency_hz'].std() / df['frequency_hz'].mean() * 100):.4f}%)")
print(f"Average Power Factor: {df['power_factor'].mean():.3f}")
print(f"Power Factor Range: {df['power_factor'].min():.3f} to {df['power_factor'].max():.3f}")

print("\n5. KEY CORRELATIONS")
print("-" * 80)
print(f"Load vs Current: {df['load_mw'].corr(df['current_a']):.3f} (expected: strong positive)")
print(f"Load vs Voltage: {df['load_mw'].corr(df['voltage_kv']):.3f} (expected: slight negative)")
print(f"Load vs Temperature: {df['load_mw'].corr(df['temperature_c']):.3f}")
print(f"Power Factor vs Reactive Power: {df['power_factor'].corr(df['reactive_power_mvar']):.3f}")

print("\n6. DATA QUALITY")
print("-" * 80)
print(f"Missing Values: {df.isnull().sum().sum()} (0%)")
print(f"Duplicate Records: {df.duplicated().sum()} (0%)")
print(f"Data Types: All appropriate (datetime + numerical)")

print("\n" + "=" * 80)
print("END OF REPORT")
print("=" * 80)

## What This Means for Electrical Engineers

### Industry Relevance:

1. **Load Forecasting**: Understanding daily and weekly patterns is essential for:
   - Generation unit commitment (deciding which generators to run)
   - Reserve margin planning (backup capacity)
   - Energy market bidding strategies

2. **Power Quality Management**: Monitoring voltage and frequency variations helps:
   - Identify grid stability issues before they become critical
   - Comply with regulatory standards (ANSI, IEEE)
   - Protect sensitive equipment from damage

3. **Demand Response Programs**: Weekday/weekend differences inform:
   - Time-of-use pricing strategies
   - Peak shaving initiatives
   - Industrial load scheduling

4. **Asset Management**: Correlation analysis between temperature and load helps:
   - Predict equipment stress during extreme weather
   - Plan maintenance during low-load periods
   - Size cooling systems appropriately

5. **Grid Planning**: Long-term pattern recognition supports:
   - Infrastructure investment decisions
   - Renewable energy integration planning
   - Capacity expansion timing

### Key Takeaways:

- **Visualization is powerful**: Charts communicate patterns that tables cannot
- **Multiple perspectives matter**: Time series, distributions, and correlations each reveal different insights
- **Statistical rigor is important**: Hypothesis testing confirms intuitions with data
- **Domain knowledge guides analysis**: Understanding power systems helps interpret findings
- **Interactive plots enhance exploration**: Plotly allows deeper investigation of patterns

### Common Mistakes:

- Relying only on summary statistics without visualizations
- Confusing correlation with causation
- Ignoring temporal patterns in time series data
- Not considering domain constraints (e.g., physical limits of equipment)
- Overlooking data quality issues during EDA

### Pro Tips:

- Always plot your data before modeling
- Look for outliers that might indicate real events vs. errors
- Consider multiple time scales (hourly, daily, weekly, seasonal)
- Use interactive plots for stakeholder presentations
- Document insights in a summary report for team communication

### Next Steps:

In the next notebook (Time Series Basics), we'll dive deeper into time-dependent patterns, stationarity testing, autocorrelation, and seasonal decomposition.